# Market and News Data Collection

Collect historical market data and current financial news for downstream research. Credentials are read from the runtime environment and are never stored in this notebook.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os
os.chdir('/content/drive/MyDrive/AAI-520-Project')

print(f"current dir: {Path.cwd()}")

Mounted at /content/drive
current dir: /content/drive/MyDrive/AAI-520-Project


In [2]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import os
os.chdir('/content/drive/MyDrive/AAI-520-Project')

print(f"current dir: {Path.cwd()}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
current dir: /content/drive/MyDrive/AAI-520-Project


In [3]:
import os
from pathlib import Path

import pandas as pd
import requests
import yfinance as yf
from dotenv import load_dotenv

project_root = Path.cwd()
if not (project_root / "pyproject.toml").exists():
    project_root = (Path.cwd() / "..").resolve()
load_dotenv(project_root / ".env")

True

In [4]:
stock_ticker = os.getenv("STOCK_TICKER", "AAPL").strip().upper()
data_period = os.getenv("DATA_PERIOD", "1y").strip()

if not stock_ticker or not data_period:
    raise ValueError("STOCK_TICKER and DATA_PERIOD are required.")

print(f"Ticker: {stock_ticker}")
print(f"Period: {data_period}")

Ticker: AAPL
Period: 3y


In [5]:
stock = yf.Ticker(stock_ticker)
stock_df = stock.history(period=data_period).reset_index()
if stock_df.empty:
    raise ValueError(f"No market data returned for {stock_ticker}.")

try:
    company_name = stock.info.get("longName", stock_ticker)
except Exception:
    company_name = stock_ticker

display(stock_df.head())
print(f"Market rows: {len(stock_df):,}")

,Date,Open,High,Low,Close,Volume,Dividends,Stock Splits
0,2023-10-16 00:00:00-04:00,174.391355,176.690264,174.154553,176.335068,52517000,0.0,0.0
1,2023-10-17 00:00:00-04:00,174.292653,176.039037,172.467349,174.785980,57549400,0.0,0.0
2,2023-10-18 00:00:00-04:00,173.236944,175.210255,172.773215,173.493469,54764400,0.0,0.0
3,2023-10-19 00:00:00-04:00,173.690792,175.466775,172.852144,173.118546,59302900,0.0,0.0
4,2023-10-20 00:00:00-04:00,172.970548,173.079081,170.336180,170.572983,64244000,0.0,0.0


Market rows: 752


In [6]:
news_api_key = os.getenv("NEWS_API_KEY")

if not news_api_key:
    raise RuntimeError(
        "NEWS_API_KEY is missing. Add NEWS_API_KEY=... to the project "
        ".env file, then restart the notebook kernel."
    )

response = requests.get(
    "https://newsapi.org/v2/everything",
    params={
        "q": stock_ticker,
        "apiKey": news_api_key,
        "language": "en",
        "pageSize": 50,
        "sortBy": "relevancy",
    },
    timeout=30,
)
response.raise_for_status()
news_df = pd.DataFrame(response.json().get("articles", []))
display(news_df.head())
print(f"News rows: {len(news_df):,}")

,source,author,title,description,url,urlToImage,publishedAt,content
0,"{'id': None, 'name': 'MacRumors'}",Joe Rossignol,"New Mac Mini Has Two Storage Downsides, Includ...","As confirmed by the website ""Expand Mac mini,""...",https://www.macrumors.com/2026/09/22/new-mac-m...,https://images.macrumors.com/t/Nn9fuxpIL2pwS6U...,2026-09-23T03:23:24Z,"As confirmed by the website ""Expand Mac mini,""..."
1,"{'id': None, 'name': '9to5Mac'}",Marcus Mendes,Apple gets a $360 price target from JP Morgan ...,Just days after Apple’s stock took a hit over ...,https://9to5mac.com/2026/10/12/apple-gets-a-36...,https://9to5mac.com/wp-content/uploads/sites/6...,2026-10-12T22:23:50Z,Just days after Apples stock took a hit over r...
2,"{'id': None, 'name': 'Yahoo Entertainment'}",Daniel Howley,Apple debuts HomeView hub as it reinvents smar...,Apple (AAPL) is reinventing its smart home str...,https://finance.yahoo.com/technology/article/a...,https://s.yimg.com/lo/mysterio/api/af74d77d03a...,2026-10-13T13:17:44Z,Apple (AAPL) is reinventing its smart home str...
3,"{'id': None, 'name': 'Yahoo Entertainment'}","Brooke DiPalma, Michael B. Kelley","Paramount-Warner Bros. finally merger closes, ...",The $110 billion merger of Paramount Skydance ...,https://finance.yahoo.com/markets/stocks/artic...,https://s.yimg.com/lo/mysterio/api/ea08c179d8a...,2026-10-06T13:03:29Z,The $110 billion merger of Paramount Skydance ...
4,"{'id': None, 'name': 'Yahoo Entertainment'}","Brooke DiPalma, Michael B. Kelley","Paramount-Warner Bros. finally merger closes, ...",The $110 billion merger of Paramount Skydance ...,https://finance.yahoo.com/markets/stocks/artic...,https://s.yimg.com/lo/mysterio/api/ea08c179d8a...,2026-10-06T13:03:29Z,The $110 billion merger of Paramount Skydance ...


News rows: 50


In [7]:
output_dir = project_root / "data" / "processed"
output_dir.mkdir(parents=True, exist_ok=True)
stock_df.to_csv(output_dir / f"market_{stock_ticker}.csv", index=False)
news_df.to_json(output_dir / f"news_{stock_ticker}.json", orient="records", date_format="iso")
print(f"Saved reusable outputs in repository root: {output_dir}")

Saved reusable outputs in repository root: /content/drive/MyDrive/AAI-520-Project/data/processed
